In [9]:
import numpy as np
import pandas as pd
from lifelines import KaplanMeierFitter
from lifelines.utils import restricted_mean_survival_time

## Read data

In [10]:
data_ct = pd.read_csv("outputs/trophos_ct_loo_covariates_diff_datasets.csv")

## Compute ATE and variance in the classical case

In [11]:
tau = 1.5

kmf_trt = KaplanMeierFitter().fit(data_ct[data_ct['TREATMENT']==1]["TIME_OF_EVENT"],data_ct[data_ct['TREATMENT']==1]["EVENT"])
rmst_trt, var_rmst_trt = restricted_mean_survival_time(kmf_trt, t=tau, return_variance=True)

kmf_con = KaplanMeierFitter().fit(data_ct[data_ct['TREATMENT']==0]["TIME_OF_EVENT"],data_ct[data_ct['TREATMENT']==0]["EVENT"])
rmst_con, var_rmst_con = restricted_mean_survival_time(kmf_con, t=tau, return_variance=True)

/Users/maylis.tran/miniforge3/envs/leaspy_v2/lib/python3.11/site-packages/lifelines/utils/__init__.py:320: IntegrationWarning: The maximum number of subdivisions (50) has been achieved.
  If increasing the limit yields no improvement it is advised to analyze 
  the integrand in order to determine the difficulties.  If the position of a 
  local difficulty can be determined (singularity, discontinuity) one will 
  probably gain from splitting up the interval and calling the integrator 
  on the subranges.  Perhaps a special-purpose integrator should be used.
  return 2 * quad(lambda tau: (tau * model.predict(tau)), 0, t, epsabs=1.49e-10, epsrel=1e-10)[0]
/Users/maylis.tran/miniforge3/envs/leaspy_v2/lib/python3.11/site-packages/lifelines/utils/__init__.py:320: IntegrationWarning: The maximum number of subdivisions (50) has been achieved.
  If increasing the limit yields no improvement it is advised to analyze 
  the integrand in order to determine the difficulties.  If the position of a 

In [12]:
# Compute classic ATE using RMST 
m = len(data_ct[data_ct['TREATMENT'] == 1])
n = len(data_ct[data_ct['TREATMENT'] == 0])

ate_classic_1 = rmst_trt - rmst_con
var_classic_1 = var_rmst_trt/m + var_rmst_con/n

var_rmst_trt_loo = data_ct[data_ct['TREATMENT'] == 1]['LOO_RMST_SEP'].var()
var_rmst_con_loo = data_ct[data_ct['TREATMENT'] == 0]['LOO_RMST_SEP'].var()

ate_classic_2 = data_ct[data_ct['TREATMENT'] == 1]['LOO_RMST_SEP'].mean() - data_ct[data_ct['TREATMENT'] == 0]['LOO_RMST_SEP'].mean()
var_classic_2 = var_rmst_trt_loo/m + var_rmst_con_loo/n

print("Average Treatment Effect (ATE) with RMST:", ate_classic_1, ", ATE with LOO RMST:", ate_classic_2)
print("Variance of ATE with RMST:", var_classic_1, ", with LOO RMST", var_classic_2)
print("Confidence interval of ATE with RMST:", ate_classic_2 - 1.96 * np.sqrt(var_classic_2), "to", ate_classic_2 + 1.96 * np.sqrt(var_classic_2))
print("Confidence interval width with RMST:", - (ate_classic_2 - 1.96 * np.sqrt(var_classic_2)) + ate_classic_2 + 1.96 * np.sqrt(var_classic_2))

Average Treatment Effect (ATE) with RMST: 0.017498977018875372 , ATE with LOO RMST: 0.01749897701791636
Variance of ATE with RMST: 0.0007930983598853449 , with LOO RMST 0.0008472646476118332
Confidence interval of ATE with RMST: -0.039552332084203445 to 0.07455028612003617
Confidence interval width with RMST: 0.11410261820423961


## Apply PPCT linear 

In [5]:
col = 'LOO_RMST_PRED_COX_PROACT_ANSWERALS'
sigma_f_2 = data_ct[col].var()
sigma_t_2 = var_rmst_trt_loo
sigma_c_2 = var_rmst_con_loo

rho_t = np.cov(
    data_ct.loc[data_ct['TREATMENT'] == 1, col],
    data_ct.loc[data_ct['TREATMENT'] == 1, 'LOO_RMST_SEP']
)[0, 1] / np.sqrt(sigma_f_2 * sigma_t_2)

rho_c = np.cov(
    data_ct.loc[data_ct['TREATMENT'] == 0, col],
    data_ct.loc[data_ct['TREATMENT'] == 0, 'LOO_RMST_SEP']
)[0, 1] / np.sqrt(sigma_f_2 * sigma_c_2)
lambda_star = 0.9393584736465697

# PPI Variance
var_ppi = ((1/m) * (sigma_t_2 + (lambda_star**2)*sigma_f_2 - 2*lambda_star*np.sqrt(sigma_f_2*sigma_t_2)*rho_t)
            + (1/n) * (sigma_c_2 + (lambda_star**2)*sigma_f_2 - 2*lambda_star*np.sqrt(sigma_f_2*sigma_c_2)*rho_c))

# PPI ATE
ate_ppi = (
    (data_ct.loc[data_ct['TREATMENT'] == 1, 'LOO_RMST_SEP'] - lambda_star * data_ct.loc[data_ct['TREATMENT'] == 1, col]).mean()
    - (data_ct.loc[data_ct['TREATMENT'] == 0, 'LOO_RMST_SEP'] - lambda_star * data_ct.loc[data_ct['TREATMENT'] == 0, col]).mean()
)

# R²
r_2 = data_ct['LOO_RMST_SEP'].corr(data_ct[col]) ** 2
var_r2 = var_classic_1 * (1 - r_2)
print(ate_ppi, var_ppi)

0.021590020259742333 0.0008232614456629458


In [6]:
results = []

for col in [c for c in data_ct.columns if c.startswith("LOO_RMST_PRED_")]:
    model_name = col.replace("LOO_RMST_PRED_", "") 
    
    sigma_f_2 = data_ct[col].var()
    sigma_t_2 = var_rmst_trt_loo
    sigma_c_2 = var_rmst_con_loo

    rho_t = np.cov(
        data_ct.loc[data_ct['TREATMENT'] == 1, col],
        data_ct.loc[data_ct['TREATMENT'] == 1, 'LOO_RMST_SEP']
    )[0, 1] / np.sqrt(sigma_f_2 * sigma_t_2)
    
    rho_c = np.cov(
        data_ct.loc[data_ct['TREATMENT'] == 0, col],
        data_ct.loc[data_ct['TREATMENT'] == 0, 'LOO_RMST_SEP']
    )[0, 1] / np.sqrt(sigma_f_2 * sigma_c_2)

    lambda_star = (n * np.sqrt(sigma_t_2) * rho_t + m * np.sqrt(sigma_c_2) * rho_c) / ((n + m) * np.sqrt(sigma_f_2))
    #lambda_star = 0.083


    # PPI Variance
    var_ppi = ((1/m) * (sigma_t_2 + (lambda_star**2)*sigma_f_2 - 2*lambda_star*np.sqrt(sigma_f_2*sigma_t_2)*rho_t)
              + (1/n) * (sigma_c_2 + (lambda_star**2)*sigma_f_2 - 2*lambda_star*np.sqrt(sigma_f_2*sigma_c_2)*rho_c))

    # PPI ATE
    ate_ppi = (
        (data_ct.loc[data_ct['TREATMENT'] == 1, 'LOO_RMST_SEP'] - lambda_star * data_ct.loc[data_ct['TREATMENT'] == 1, col]).mean()
        - (data_ct.loc[data_ct['TREATMENT'] == 0, 'LOO_RMST_SEP'] - lambda_star * data_ct.loc[data_ct['TREATMENT'] == 0, col]).mean()
    )

    # R²
    r_2 = data_ct['LOO_RMST_SEP'].corr(data_ct[col]) ** 2
    var_r2 = var_classic_1 * (1 - r_2)

    # Append results
    results.append({
        "Model": model_name,
        "PPI_ATE": ate_ppi,
        "PPI_Variance": var_ppi,
        "Lambda_star": lambda_star,
        "R2": r_2,
        "PPI_Var_R2_formula": var_r2,
        "Number of patients possibl to remove": int(r_2*(m + n))
    })

df_ppi_summary = pd.DataFrame(results).set_index("Model")
df_ppi_summary = df_ppi_summary.sort_values(by="PPI_Variance", ascending=True)
df_ppi_summary


,PPI_ATE,PPI_Variance,Lambda_star,R2,PPI_Var_R2_formula,Number of patients possibl to remove
Model,,,,,,
POLY_PROACT,0.018508,0.000744,0.881053,0.119990,0.000698,60
POLY_PROACT_PULSE,0.018761,0.000747,0.904668,0.116386,0.000701,59
POLY_PROACT_ANSWERALS,0.019001,0.000752,0.953947,0.110515,0.000705,56
POLY_PROACT_PULSE_ANSWERALS,0.019126,0.000753,0.941759,0.108811,0.000707,55
POLY_PROACT_PULSE_NEUROBANK,0.017389,0.000762,0.915670,0.098812,0.000715,50
POLY_PROACT_NEUROBANK,0.017319,0.000763,0.914879,0.097804,0.000716,49
POLY_PROACT_PULSE_NEUROBANK_ANSWERALS,0.017369,0.000765,0.916483,0.095697,0.000717,48
POLY_PROACT_NEUROBANK_ANSWERALS,0.017278,0.000766,0.915592,0.094471,0.000718,47
POLY_PULSE_ANSWERALS,0.019702,0.000773,0.860406,0.086196,0.000725,43


In [7]:
print(df_ppi_summary.to_latex(escape=True))

\begin{tabular}{lrrrrrr}
\toprule
 & PPI\_ATE & PPI\_Variance & Lambda\_star & R2 & PPI\_Var\_R2\_formula & Number of patients possibl to remove \\
Model &  &  &  &  &  &  \\
\midrule
POLY\_PROACT & 0.018508 & 0.000744 & 0.881053 & 0.119990 & 0.000698 & 60 \\
POLY\_PROACT\_PULSE & 0.018761 & 0.000747 & 0.904668 & 0.116386 & 0.000701 & 59 \\
POLY\_PROACT\_ANSWERALS & 0.019001 & 0.000752 & 0.953947 & 0.110515 & 0.000705 & 56 \\
POLY\_PROACT\_PULSE\_ANSWERALS & 0.019126 & 0.000753 & 0.941759 & 0.108811 & 0.000707 & 55 \\
POLY\_PROACT\_PULSE\_NEUROBANK & 0.017389 & 0.000762 & 0.915670 & 0.098812 & 0.000715 & 50 \\
POLY\_PROACT\_NEUROBANK & 0.017319 & 0.000763 & 0.914879 & 0.097804 & 0.000716 & 49 \\
POLY\_PROACT\_PULSE\_NEUROBANK\_ANSWERALS & 0.017369 & 0.000765 & 0.916483 & 0.095697 & 0.000717 & 48 \\
POLY\_PROACT\_NEUROBANK\_ANSWERALS & 0.017278 & 0.000766 & 0.915592 & 0.094471 & 0.000718 & 47 \\
POLY\_PULSE\_ANSWERALS & 0.019702 & 0.000773 & 0.860406 & 0.086196 & 0.000725 & 43 \\
POLY\_